# 09 — Capstone: policy-derived ground truth

        **Estimated time:** 50 minutes<br>
        **Prerequisites:** 08 — Local MLflow evidence and promotion decisions<br>
        **Learner-produced evidence:** a versioned 400/100/150 policy dataset and deterministic ceiling

        ## Learning objectives

        - Separate deterministic, policy, external-lookup, and human-judgment rules.
- Generate controlled ground truth without asking an LLM to invent critical labels.
- Verify capstone split counts, slice coverage, hashes, and the deterministic ceiling.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Why Kaggle is not the capstone ground truth

Customer-support records do not represent Databricks application
readiness. The capstone uses a small reviewed domain dataset generated
from an explicit policy engine. Every expected check carries rule kind,
source fields, facts origin, severity, and remediation provenance.


In [ ]:
from collections import Counter

import pandas as pd

from aai_local_finetuning.capstone import (
    REQUIRED_FROZEN_TEST_SLICES,
    deterministic_capstone_predictions,
    evaluate_capstone_predictions,
    evaluate_manifest,
    generate_capstone_dataset,
    load_capstone_records,
    render_capstone_mlx_dataset,
    rule_catalog,
)
from aai_local_finetuning.settings import PROJECT_ROOT

rules = rule_catalog()
pd.DataFrame(
    [
        {
            "rule": rule.rule_id,
            "kind": rule.kind.value,
            "severity": rule.failure_severity.value,
            "source_fields": ", ".join(rule.source_fields),
        }
        for rule in rules
    ]
)

## Rule kinds define authority

Deterministic rules read manifest facts. Policy rules apply a versioned
threshold or vocabulary. External lookups require an authorized system;
human judgment requires a person. The latter two route to review rather
than letting a tiny model invent facts.


In [ ]:
Counter(rule.kind.value for rule in rules)

## Generate reviewed combinations

A fixed seed creates controlled rule violations and interacting failures.
The frozen test includes required slices and unseen combinations of known
failures. Generation writes portable records and immutable hashes.


In [ ]:
source_dir = PROJECT_ROOT / "data" / "processed" / "capstone-readiness-v1"
mlx_dir = PROJECT_ROOT / "data" / "processed" / "capstone-mlx-v1"
split_manifest = generate_capstone_dataset(source_dir)
training_manifest = render_capstone_mlx_dataset(source_dir, mlx_dir)
{
    "counts": {
        artifact.split.value: artifact.record_count
        for artifact in split_manifest.artifacts
    },
    "frozen_test": split_manifest.frozen_test,
    "dataset_sha256": split_manifest.dataset_sha256,
    "portable_training_fingerprint": training_manifest.dataset_fingerprint,
}

## Inspect one synthetic example and its provenance

This is generated application metadata, not customer data. The expected
review comes entirely from the policy engine, including review routing
for facts that are unavailable locally.


In [ ]:
train_records = load_capstone_records(source_dir / "train.jsonl")
example = train_records[0]
{
    "example_id": example.example_id,
    "slices": example.metadata.slices,
    "manifest": example.manifest,
    "expected_status": example.expected_output.status.value,
    "non_pass_checks": [
        {
            "name": check.name,
            "result": check.result.value,
            "severity": check.severity.value,
            "rule_kind": check.provenance.rule_kind.value,
            "facts_origin": check.provenance.facts_origin,
        }
        for check in example.expected_output.checks
        if check.result.value != "pass"
    ],
}

## Verify the frozen slice contract and deterministic ceiling

Every required slice must occur. The policy engine is the accuracy ceiling
for rules it fully determines, so replacing those decisions with a model
cannot improve correctness.


In [ ]:
test_records = load_capstone_records(source_dir / "test.jsonl")
test_slices = Counter(
    slice_name for record in test_records for slice_name in record.metadata.slices
)
missing_slices = sorted(set(REQUIRED_FROZEN_TEST_SLICES) - set(test_slices))
policy_report = evaluate_capstone_predictions(
    test_records,
    deterministic_capstone_predictions(test_records),
)
{
    "missing_required_slices": missing_slices,
    "exact_review_rate": policy_report.aggregate.exact_review_rate,
    "schema_validity": policy_report.aggregate.schema_validity_rate,
    "slice_counts": dict(sorted(test_slices.items())),
}

## Exercise — review a new manifest

Change one field and inspect which check changes. Success means you can
explain whether the outcome came from a manifest fact, platform policy,
an external system, or human review.


In [ ]:
learner_manifest = dict(example.manifest)
learner_manifest["owner"] = None
learner_review = evaluate_manifest(learner_manifest)
[
    {
        "name": check.name,
        "result": check.result.value,
        "kind": check.provenance.rule_kind.value,
        "origin": check.provenance.facts_origin,
        "evidence": check.evidence,
    }
    for check in learner_review.checks
    if check.result.value != "pass"
]

**Hint:** the engine may explain an absent local fact, but it must not
claim that a registry lookup or human review occurred when it did not.


## Checkpoint

You now have deterministic, versioned ground truth and a measurable
ceiling—not LLM-generated labels presented as facts.

**Next:** `10_capstone_model_vs_hybrid.ipynb` tests where a tiny model may
add value without owning authoritative readiness decisions.
